# Synthetic Regression Data
:label:`sec_synthetic-regression-data`


Machine learning is all about extracting information from data.
So you might wonder, what could we possibly learn from synthetic data?
While we might not care intrinsically about the patterns 
that we ourselves baked into an artificial data generating model,
such datasets are nevertheless useful for didactic purposes,
helping us to evaluate the properties of our learning 
algorithms and to confirm that our implementations work as expected.
For example, if we create data for which the correct parameters are known *a priori*,
then we can check that our model can in fact recover them.


In [1]:
%matplotlib inline
import random
import torch
from d2l import torch as d2l

## Generating the Dataset

For this example, we will work in low dimension
for succinctness.
The following code snippet generates 1000 examples
with 2-dimensional features drawn 
from a standard normal distribution.
The resulting design matrix $\mathbf{X}$
belongs to $\mathbb{R}^{1000 \times 2}$. 
We generate each label by applying 
a *ground truth* linear function, 
corrupting them via additive noise $\boldsymbol{\epsilon}$, 
drawn independently and identically for each example:

(**$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \boldsymbol{\epsilon}.$$**)

For convenience we assume that $\boldsymbol{\epsilon}$ is drawn 
from a normal distribution with mean $\mu= 0$ 
and standard deviation $\sigma = 0.01$.
Note that for object-oriented design
we add the code to the `__init__` method of a subclass of `d2l.DataModule` (introduced in :numref:`oo-design-data`). 
It is good practice to allow the setting of any additional hyperparameters. 
We accomplish this with `save_hyperparameters()`. 
The `batch_size` will be determined later.


In [2]:
class SyntheticRegressionData(d2l.DataModule):  #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n, 1) * noise
        self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise

Below, we set the true parameters to $\mathbf{w} = [2, -3.4]^\top$ and $b = 4.2$.
Later, we can check our estimated parameters against these *ground truth* values.


In [3]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)

[**Each row in `features` consists of a vector in $\mathbb{R}^2$ and each row in `labels` is a scalar.**] Let's have a look at the first entry.


In [4]:
print('features:', data.X[0],'\nlabel:', data.y[0])

features: tensor([ 0.1966, -0.6143]) 
label: tensor([6.6860])


## Reading the Dataset

Training machine learning models often requires multiple passes over a dataset, 
grabbing one minibatch of examples at a time. 
This data is then used to update the model. 
To illustrate how this works, we 
[**implement the `get_dataloader` method,**] 
registering it in the `SyntheticRegressionData` class via `add_to_class` (introduced in :numref:`oo-design-utilities`).
It (**takes a batch size, a matrix of features,
and a vector of labels, and generates minibatches of size `batch_size`.**)
As such, each minibatch consists of a tuple of features and labels. 
Note that we need to be mindful of whether we're in training or validation mode: 
in the former, we will want to read the data in random order, 
whereas for the latter, being able to read data in a pre-defined order 
may be important for debugging purposes.


In [5]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        # The examples are read in random order
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train+self.num_val))
    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i: i+self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

To build some intuition, let's inspect the first minibatch of
data. Each minibatch of features provides us with both its size and the dimensionality of input features.
Likewise, our minibatch of labels will have a matching shape given by `batch_size`.


In [6]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


While seemingly innocuous, the invocation 
of `iter(data.train_dataloader())` 
illustrates the power of Python's object-oriented design. 
Note that we added a method to the `SyntheticRegressionData` class
*after* creating the `data` object. 
Nonetheless, the object benefits from 
the *ex post facto* addition of functionality to the class.

Throughout the iteration we obtain distinct minibatches
until the entire dataset has been exhausted (try this).
While the iteration implemented above is good for didactic purposes,
it is inefficient in ways that might get us into trouble with real problems.
For example, it requires that we load all the data in memory
and that we perform lots of random memory access.
The built-in iterators implemented in a deep learning framework
are considerably more efficient and they can deal
with sources such as data stored in files, 
data received via a stream, 
and data generated or processed on the fly. 
Next let's try to implement the same method using built-in iterators.

## Concise Implementation of the Data Loader

Rather than writing our own iterator,
we can [**call the existing API in a framework to load data.**]
As before, we need a dataset with features `X` and labels `y`. 
Beyond that, we set `batch_size` in the built-in data loader 
and let it take care of shuffling examples  efficiently.


In [7]:
@d2l.add_to_class(d2l.DataModule)  #@save
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size,
                                       shuffle=train)

In [8]:
@d2l.add_to_class(SyntheticRegressionData)  #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

The new data loader behaves just like the previous one, except that it is more efficient and has some added functionality.


In [9]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


For instance, the data loader provided by the framework API 
supports the built-in `__len__` method, 
so we can query its length, 
i.e., the number of batches.


In [10]:
len(data.train_dataloader())

32

## Summary

Data loaders are a convenient way of abstracting out 
the process of loading and manipulating data. 
This way the same machine learning *algorithm* 
is capable of processing many different types and sources of data 
without the need for modification. 
One of the nice things about data loaders 
is that they can be composed. 
For instance, we might be loading images 
and then have a postprocessing filter 
that crops them or modifies them in other ways. 
As such, data loaders can be used 
to describe an entire data processing pipeline. 

As for the model itself, the two-dimensional linear model 
is about the simplest we might encounter. 
It lets us test out the accuracy of regression models 
without worrying about having insufficient amounts of data 
or an underdetermined system of equations. 
We will put this to good use in the next section.  


## Exercises

1. What will happen if the number of examples cannot be divided by the batch size. How would you change this behavior by specifying a different argument by using the framework's API?
1. Suppose that we want to generate a huge dataset, where both the size of the parameter vector `w` and the number of examples `num_examples` are large.
    1. What happens if we cannot hold all data in memory?
    1. How would you shuffle the data if it is held on disk? Your task is to design an *efficient* algorithm that does not require too many random reads or writes. Hint: [pseudorandom permutation generators](https://en.wikipedia.org/wiki/Pseudorandom_permutation) allow you to design a reshuffle without the need to store the permutation table explicitly :cite:`Naor.Reingold.1999`. 
1. Implement a data generator that produces new data on the fly, every time the iterator is called. 
1. How would you design a random data generator that generates *the same* data each time it is called?


#### 1. What will happen if the number of examples cannot be divided by the batch size. How would you change this behavior by specifying a different argument by using the framework's API?

##### Answer

When the number of examples cannot be divided evenly by the batch size, by default, the last batch will contain fewer examples than the specified batch size. This happens because PyTorch's DataLoader will include all examples in the dataset, even if it means creating a smaller final batch.

For example, if you have 1025 examples and a batch size of 32:
- You'll get 32 full batches (32 × 32 = 1024 examples)
- Plus one small batch with just 1 example

This behavior can sometimes be problematic:
1. The smaller batch might cause issues with batch normalization layers
2. Some operations expect consistent tensor dimensions
3. It can lead to inefficient GPU utilization

To modify this behavior using PyTorch's API, you can use the `drop_last` parameter when creating the DataLoader:

```python
torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

2. Suppose that we want to generate a huge dataset, where both the size of the parameter vector `w` and the number of examples `num_examples` are large.
    1. What happens if we cannot hold all data in memory?
    1. How would you shuffle the data if it is held on disk? Your task is to design an *efficient* algorithm that does not require too many random reads or writes. Hint: [pseudorandom permutation generators](https://en.wikipedia.org/wiki/Pseudorandom_permutation) allow you to design a reshuffle without the need to store the permutation table explicitly :cite:`Naor.Reingold.1999`. 

n
### 2. Handling Huge Datasets with Large Parameter Vectors and Examples

#### 2.1 What happens if we cannot hold all data in memory?

When dealing with datasets too large to fit in memory, several challenges and solutions arise:

1. **Memory Overflow**: Attempting to load the entire dataset at once would cause memory errors or system crashes.

2. **Solutions for Large Dataset Training**:

   a) **Data Streaming/Generators**: Instead of loading all data at once, we can implement data generators that load small portions of data on-demand.
   
   ```python
   class DiskDataset:
       def __init__(self, file_path, batch_size):
           self.file_path = file_path
           self.batch_size = batch_size
           # Get dataset size without loading all data
           self.size = self._get_dataset_size()
           
       def __iter__(self):
           with open(self.file_path, 'rb') as f:
               for i in range(0, self.size, self.batch_size):
                   # Read just enough data for one batch
                   batch_data = self._read_batch(f, i)
                   yield batch_data
   ```
   
   b) **Memory-Mapped Files**: Use memory mapping (like NumPy's `memmap` or PyTorch's `MemoryMappedDataset`) to access disk-stored data as if it were in memory.
   
   ```python
   import numpy as np
   
   # Create memory-mapped array
   X_mmap = np.memmap('features.dat', dtype='float32', mode='r', 
                      shape=(num_examples, feature_dim))
   y_mmap = np.memmap('labels.dat', dtype='float32', mode='r',
                      shape=(num_examples, 1))
   ```
   
   c) **Out-of-Core Learning**: Algorithms specifically designed to learn from data that doesn't fit in memory by processing it in chunks.
   
   d) **Selective Loading**: Only load the subset of features needed for the current computation, especially effective when `w` is sparse.

#### 2.2 How would you shuffle the data if it is held on disk?

Shuffling data on disk efficiently requires avoiding random access patterns, which are extremely slow. Here's an efficient approach using pseudorandom permutation generators:

1. **Using Pseudorandom Permutation Generators**:
   
   Instead of physically shuffling data (which would require many random reads and writes), we can use a function that deterministically maps indices i → π(i) where π is a permutation. This creates the effect of shuffling without actually rearranging the data.
   
   ```python
   class DiskShuffledDataset:
       def __init__(self, file_path, batch_size, seed=42):
           self.file_path = file_path
           self.batch_size = batch_size
           self.size = self._get_dataset_size()
           self.seed = seed
           
       def _permutation_function(self, idx):
           """Feistel network-based permutation function"""
           # Using a simple encryption-like function
           a, b = idx % 65536, idx // 65536
           # Multiple rounds of mixing
           for i in range(4):
               # Hash function with seed
               hash_val = (a * 31 + self.seed * 17 + i) % 65536
               a, b = b, (a ^ hash_val) % 65536
           return a + b * 65536
       
       def __iter__(self):
           # Generate shuffled indices using the permutation function
           indices = [self._permutation_function(i) % self.size 
                     for i in range(self.size)]
           
           # Read batches according to these indices
           for i in range(0, self.size, self.batch_size):
               batch_indices = indices[i:i+self.batch_size]
               # Sort to improve disk access locality
               batch_indices.sort()
               # Read the batch using these indices
               batch_data = self._read_batch_indices(batch_indices)
               yield batch_data
   ```

2. **Block-wise Shuffling**:
   
   Another approach is to divide the dataset into blocks that fit in memory:
   
   a) Shuffle each block internally in memory
   b) Write shuffled blocks back to disk
   c) Perform a multi-way merge with randomized selection
   
   This algorithm limits random access while still providing good shuffling quality.

3. **External Memory Algorithms**:
   
   For extremely large datasets, you could use external sorting algorithms adapted for shuffling, or database-like approaches with indexed chunks.

The pseudorandom permutation approach is particularly elegant because:

- It requires zero additional storage for the permutation
- It gives a perfect shuffle
- It allows deterministic reproduction of the shuffle given the same seed
- It avoids random disk I/O entirely by computing shuffled indices on-the-fly

This approach achieves the perfect balance between shuffle quality and disk access efficiency, making it ideal for handling extremely large datasets.
```

3. Implement a data generator that produces new data on the fly, every time the iterator is called. 


```markdown
#### 3. Implement a data generator that produces new data on the fly, every time the iterator is called.

##### Answer

To implement a data generator that produces completely new synthetic data on each iteration (rather than just reshuffling existing data), we need to create a custom implementation that generates fresh data samples each time it's called.

Here's a complete implementation of an on-the-fly data generator using the PyTorch framework:

```python
class OnTheFlyRegressionData(d2l.DataModule):
    """Generates synthetic regression data on the fly for each iteration."""
    def __init__(self, w, b, noise=0.01, batch_size=32, num_batches=32):
        super().__init__()
        self.save_hyperparameters()
        self.w = w
        self.b = b
        self.noise = noise
        self.batch_size = batch_size
        self.num_batches = num_batches  # Control how many batches per epoch
        
    def get_dataloader(self, train):
        return self if train else self  # Same generator for train and val
    
    def __iter__(self):
        """Generate fresh data for each iteration cycle."""
        self.current_batch = 0
        return self
    
    def __next__(self):
        """Generate a new batch of synthetic data on each call."""
        if self.current_batch >= self.num_batches:
            raise StopIteration
            
        # Generate new features for this batch
        X = torch.randn(self.batch_size, len(self.w))
        
        # Generate corresponding labels using the true model
        noise = torch.randn(self.batch_size, 1) * self.noise
        y = torch.matmul(X, self.w.reshape((-1, 1))) + self.b + noise
        
        self.current_batch += 1
        return X, y
    
    def __len__(self):
        """Return the number of batches per epoch."""
        return self.num_batches
```

This implementation:
1. Creates a class that inherits from `d2l.DataModule` but also implements the iterator protocol
2. Generates completely new data for each batch on each iteration
3. Allows control over how many batches per epoch via `num_batches`
4. Works with the training framework by implementing the DataLoader interface

Usage example:
```python
# Create the on-the-fly data generator with true parameters w=[2, -3.4], b=4.2
data = OnTheFlyRegressionData(w=torch.tensor([2, -3.4]), b=4.2, batch_size=32, num_batches=50)

# Use it in training
for epoch in range(3):
    print(f"Epoch {epoch}")
    for X, y in data.train_dataloader():
        # Each batch contains fresh data following the same distribution
        # Use X, y for model training
        pass
```

The key advantage is that we're never storing the entire dataset in memory - we just generate each batch on demand using the parameter vector `w` and bias `b`. This can be extremely memory-efficient for large-scale synthetic data experiments.
```

4. How would you design a random data generator that generates *the same* data each time it is called?

Answer: I will use a seed value.

You're on the right track with using a seed. Let me expand on that idea with a more complete implementation.

```markdown
#### 4. How would you design a random data generator that generates *the same* data each time it is called?

##### Answer

Using a seed value is exactly the right approach! When we set a fixed seed for random number generators, they will produce the same sequence of "random" numbers every time they're initialized with that seed.

Here's a complete implementation of a reproducible data generator:

```python
class ReproducibleRegressionData(d2l.DataModule):
    """Generates the same synthetic regression data on each iteration."""
    def __init__(self, w, b, noise=0.01, batch_size=32, num_batches=32, seed=42):
        super().__init__()
        self.save_hyperparameters()
        self.w = w
        self.b = b
        self.noise = noise
        self.batch_size = batch_size
        self.num_batches = num_batches
        self.seed = seed
        
    def get_dataloader(self, train):
        return self
    
    def __iter__(self):
        """Reset the random generator to produce the same sequence."""
        # Reset random state to always generate the same sequence
        torch.manual_seed(self.seed)
        random.seed(self.seed)
        np.random.seed(self.seed)
        
        self.current_batch = 0
        return self
    
    def __next__(self):
        """Generate the next batch in the deterministic sequence."""
        if self.current_batch >= self.num_batches:
            raise StopIteration
            
        # These will be the same for each epoch due to the seed
        X = torch.randn(self.batch_size, len(self.w))
        noise = torch.randn(self.batch_size, 1) * self.noise
        y = torch.matmul(X, self.w.reshape((-1, 1))) + self.b + noise
        
        self.current_batch += 1
        return X, y
    
    def __len__(self):
        return self.num_batches
```

Key aspects of this implementation:

1. **Fixed Seed**: We set a default seed of 42, but allow it to be specified as a parameter.

2. **Seed Reset**: In the `__iter__` method, we reset all relevant random number generators (PyTorch, Python's random, and NumPy) to the same seed value at the beginning of each iteration cycle.

3. **Deterministic Generation**: Because we reset the random state before generating each epoch's data, the sequence of batches will be identical in every epoch.

This is particularly useful for:

- **Debugging**: You can reproduce the exact same conditions to isolate issues
- **Research Reproducibility**: Others can exactly replicate your experimental conditions
- **Controlled Experiments**: When comparing different models or hyperparameters, you can ensure they're trained on identical data

The approach works because pseudorandom number generators are actually deterministic algorithms that produce sequences that appear random but are completely determined by their starting state (the seed).

If needed, you could also implement variations:
- Different seeds for different epochs but consistent across runs
- Same data order but different mini-batch compositions
- Deterministic training data but random validation data

This pattern is extremely common in machine learning experimentation to ensure scientific reproducibility.
```

[Discussions](https://discuss.d2l.ai/t/6663)
